# KEIBA MASTER - XGBoost MLモデル学習

Vercelの学習データAPIからデータを取得し、XGBoostで勝率/複勝率予測モデルを学習します。

## 使い方
1. 下の「設定」セルで `VERCEL_URL` と `SYNC_KEY` を入力
2. 「ランタイム → すべて実行」
3. 最後に3ファイルがダウンロードされるので `model/` に配置してコミット

In [ ]:
# ==================== 設定 ====================
# ↓↓↓ ここを自分の環境に合わせて変更 ↓↓↓

VERCEL_URL = "https://your-app.vercel.app"  # VercelデプロイURL
SYNC_KEY = ""  # SYNC_KEY環境変数の値（未設定なら空文字列）

# データ取得範囲
DATE_FROM = "2020-01-01"
DATE_TO = "2099-12-31"

In [ ]:
# ==================== インストール ====================
!pip install xgboost scikit-learn requests matplotlib -q

import json
import requests
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, classification_report
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

print('OK')

In [ ]:
# ==================== データ取得 ====================
headers = {}
if SYNC_KEY:
    headers['x-sync-key'] = SYNC_KEY

print(f'データ取得中: {VERCEL_URL}/api/ml-export?from={DATE_FROM}&to={DATE_TO}')
resp = requests.get(
    f'{VERCEL_URL}/api/ml-export?from={DATE_FROM}&to={DATE_TO}',
    headers=headers,
    timeout=120,
)
resp.raise_for_status()
data = resp.json()

feature_names = data['feature_names']
rows = data['rows']
stats = data.get('stats', {})

print(f'\n=== データ概要 ===')
print(f'総サンプル数: {len(rows)}')
print(f'レース数: {stats.get("races", "?")}')
print(f'特徴量数: {len(feature_names)}')
print(f'特徴量: {feature_names}')

if len(rows) == 0:
    raise ValueError('学習データが0件です。\n'
                     '予想生成済み + 結果確定済みのレースが必要です。\n'
                     'バルクインポートの予想フェーズが完了しているか確認してください。')

# NumPy配列に変換
X = np.array([r['features'] for r in rows], dtype=np.float32)
y_win = np.array([r['label_win'] for r in rows], dtype=np.int8)
y_place = np.array([r['label_place'] for r in rows], dtype=np.int8)

print(f'\n勝率: {y_win.mean():.4f} ({y_win.sum()}勝)')
print(f'複勝率: {y_place.mean():.4f} ({y_place.sum()}回)')
print(f'ユニークレース: {len(set(r["race_id"] for r in rows))}')

In [ ]:
# ==================== 時系列分割 ====================
# race_idは日付順にソート可能（YYYYMMDD...形式）
race_ids = [r['race_id'] for r in rows]
sorted_indices = sorted(range(len(rows)), key=lambda i: race_ids[i])

# 最新 15% をバリデーションに使用
split_idx = int(len(sorted_indices) * 0.85)
train_idx = sorted_indices[:split_idx]
val_idx = sorted_indices[split_idx:]

X_train, X_val = X[train_idx], X[val_idx]
y_win_train, y_win_val = y_win[train_idx], y_win[val_idx]
y_place_train, y_place_val = y_place[train_idx], y_place[val_idx]

print(f'学習: {len(train_idx)}件, 検証: {len(val_idx)}件')
print(f'学習勝率: {y_win_train.mean():.4f}, 検証勝率: {y_win_val.mean():.4f}')

In [ ]:
# ==================== 勝利モデル学習 ====================
win_rate = y_win_train.mean()
scale_win = (1 - win_rate) / win_rate if win_rate > 0 else 10

model_win = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_win,
    eval_metric='auc',
    early_stopping_rounds=30,
    random_state=42,
)

model_win.fit(
    X_train, y_win_train,
    eval_set=[(X_val, y_win_val)],
    verbose=50,
)

win_probs = model_win.predict_proba(X_val)[:, 1]
win_auc = roc_auc_score(y_win_val, win_probs)
print(f'\n=== 勝利モデル AUC: {win_auc:.4f} ===')
print('(0.50 = ランダム, 0.65+ = 有効, 0.70+ = 優秀)')
print(classification_report(y_win_val, model_win.predict(X_val), target_names=['敗北', '勝利']))

In [ ]:
# ==================== 複勝モデル学習 ====================
place_rate = y_place_train.mean()
scale_place = (1 - place_rate) / place_rate if place_rate > 0 else 3

model_place = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_place,
    eval_metric='auc',
    early_stopping_rounds=30,
    random_state=42,
)

model_place.fit(
    X_train, y_place_train,
    eval_set=[(X_val, y_place_val)],
    verbose=50,
)

place_probs = model_place.predict_proba(X_val)[:, 1]
place_auc = roc_auc_score(y_place_val, place_probs)
print(f'\n=== 複勝モデル AUC: {place_auc:.4f} ===')

In [ ]:
# ==================== 特徴量重要度 ====================
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, model, title in [(axes[0], model_win, '勝利モデル'), (axes[1], model_place, '複勝モデル')]:
    importances = model.feature_importances_
    sorted_idx = np.argsort(importances)
    ax.barh([feature_names[i] for i in sorted_idx], importances[sorted_idx])
    ax.set_title(f'{title} - Feature Importance')
    ax.set_xlabel('Importance')

plt.tight_layout()
plt.show()

In [ ]:
# ==================== モデル保存 & ダウンロード ====================
import os
os.makedirs('model', exist_ok=True)

model_win.save_model('model/xgb_win.json')
model_place.save_model('model/xgb_place.json')

with open('model/feature_names.json', 'w', encoding='utf-8') as f:
    json.dump(feature_names, f, ensure_ascii=False, indent=2)

print(f'\n=== 結果サマリー ===')
print(f'学習データ: {len(rows)}件 ({len(set(r["race_id"] for r in rows))}レース)')
print(f'勝利モデル AUC: {win_auc:.4f}')
print(f'複勝モデル AUC: {place_auc:.4f}')
print(f'\nファイル保存完了:')
print(f'  model/xgb_win.json   ({os.path.getsize("model/xgb_win.json") / 1024:.0f} KB)')
print(f'  model/xgb_place.json ({os.path.getsize("model/xgb_place.json") / 1024:.0f} KB)')
print(f'  model/feature_names.json')

# Colabからダウンロード
from google.colab import files
files.download('model/xgb_win.json')
files.download('model/xgb_place.json')
files.download('model/feature_names.json')

print('\nダウンロードした3ファイルを keiba/model/ に配置してコミットしてください')